# 11장 PyTorch 첫걸음 — 텐서, autograd, nn.Module

**3단계 · 다시 만들기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_3_build.ipynb)

이 노트북에는 **주석만 남긴 코드**가 들어 있습니다. 책과 1단계 노트북을 덮고, 주석을 실마리 삼아 코드를 되살려 보세요. 아래 표의 「다시 만들기」 과제가 이 노트북의 목표입니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 채운 뒤 셀을 실행해 저장하고, 확인 셀로 돌려 봅니다.
- 막히면 오류 메시지를 먼저 읽고(부록 A), 그래도 안 되면 1단계 노트북에서 그 부분만 확인하세요.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | 세 파일을 한 줄씩 치고 출력의 **shape** 이 같은지 대조 (난수 값은 달라도 됨) | [ch11_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_1_follow.ipynb) |
| 2. 주석 달기 | `p03_module.py` 의 `MLP` 에 각 줄이 "층을 만드는 줄"인지 "계산 순서를 적는 줄"인지 표시 | [ch11_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `MLP` 를 복원하고, `hidden` 을 64로 바꿔 파라미터 수를 예측·확인 | [ch11_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_3_build.ipynb) **(지금 이 노트북)** |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "11"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/11")

## `p03_module.py` — 대상: `MLP`

In [ ]:
%%writefile p03_module.py
# 11.4 nn.Module: 가중치를 가진 함수를 클래스로
import torch
import torch.nn as nn

torch.manual_seed(0)                                   # 난수 씨앗 고정 (결과 재현용)

# ---- 가장 작은 층: 선형 층 y = x @ W^T + b
linear = nn.Linear(3, 2)                               # 입력 3개 → 출력 2개
print(linear)
print("weight shape:", linear.weight.shape, "| bias shape:", linear.bias.shape)
x = torch.ones(4, 3)                                   # 배치 4개, 각 3차원
print("출력 shape:", linear(x).shape)                   # [4, 2]

# 가중치는 파라미터(학습 대상)이고 requires_grad=True 다
for name, p in linear.named_parameters():
    print(f"  {name}: {tuple(p.shape)}, requires_grad={p.requires_grad}")


# ---- 내 모델: 4장의 클래스·상속이 그대로
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        # 부모(nn.Module)의 준비를 먼저
        # 층을 속성으로 두면 자동으로 등록된다
        raise NotImplementedError("여기를 채우세요")

    def forward(self, x):                              # 계산 순서만 적는다
        # 비선형: 음수를 0으로
        raise NotImplementedError("여기를 채우세요")


model = MLP(in_dim=784, hidden=128, out_dim=10)        # MNIST용: 784 픽셀 → 10개 숫자
print(model)

# 파라미터 수 세기
n_params = sum(p.numel() for p in model.parameters())
print("파라미터 수:", n_params)                         # 784*128+128 + 128*10+10 = 101,770

# 배치 하나를 통과시켜 본다
imgs = torch.rand(64, 1, 28, 28)                       # 가짜 이미지 64장
out = model(imgs.flatten(1))                           # [64, 784] → [64, 10]
print("출력 shape:", out.shape)
print("첫 이미지의 점수 10개:", [round(v, 2) for v in out[0].tolist()])
print("예측(가장 큰 점수의 자리):", out[0].argmax().item())

# state_dict: 가중치를 담은 딕셔너리 (3장) — 13장에서 저장할 것
sd = model.state_dict()
print("state_dict 키:", list(sd.keys()))
print("fc1.weight shape:", sd["fc1.weight"].shape)

# 흔한 실수: 모양이 안 맞으면 바로 오류
try:
    model(imgs)                                        # [64,1,28,28] 을 그대로 넣으면?
except RuntimeError as e:
    print("RuntimeError:", str(e)[:90], "...")

### 확인

In [ ]:
!python p03_module.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- p03_module.py

## 그다음

`hidden` 을 64로 바꾸면 파라미터가 몇 개가 될지 먼저 계산하고, 실행해 확인하세요.

In [ ]:
# 여기에 이어서 작성하세요